# Q-MagOpt walkthrough

Constraint-aware quantum optimization for information-aware magnetic navigation.

This notebook walks through the four ideas the repository is built on:

1. Position information from a magnetic map is **Fisher information**, and a single scalar reading is rank one.
2. Route choice therefore changes the achievable position accuracy, measurably, in metres.
3. Exact D-optimality (`det J`) is **already quadratic** in the selection variables, so it needs no surrogate.
4. An XY mixer keeps QAOA inside the feasible one-hot subspace -- and the comparison has to be conditional to be fair.


In [ ]:
# Works whether or not the package is pip-installed.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## 1. The map and its information content


In [ ]:
from dataclasses import replace

import numpy as np

from qmagopt import build_demo_problem, information_summary

problem = build_demo_problem(seed=7)
mmap = problem.magnetic_map

print(f"grid            {mmap.shape[0]} rows x {mmap.shape[1]} steps, {mmap.cell_size_m:.0f} m cells")
print(
    f"sensor sigma    {mmap.sensor.sigma_nt:.2f} nT  "
    f"(noise {mmap.sensor.noise_nt} nT, map error {mmap.sensor.map_error_nt} nT)"
)
print(f"anomaly range   {mmap.field_nt.min():.0f} .. {mmap.field_nt.max():.0f} nT")
information_summary(problem)

A single scalar total-field measurement constrains position only along the local
gradient direction, so its Fisher matrix is rank one. Position becomes observable
only by accumulating measurements whose gradients point in *different* directions.


In [ ]:
J_cell = mmap.fisher[1, 1]
print("per-cell Fisher matrix (1/m^2):")
print(J_cell)
print("rank:", np.linalg.matrix_rank(J_cell, tol=1e-12))

## 2. Route choice moves the position CRLB


In [ ]:
from qmagopt import position_crlb_m

print(f"dead reckoning alone: {position_crlb_m(mmap.prior_fisher):>6.1f} m")
for route in sorted(problem.feasible_routes(), key=problem.route_crlb_m)[:3]:
    print(f"  best  {route}  {problem.route_crlb_m(route):>6.1f} m")
for route in sorted(problem.feasible_routes(), key=problem.route_crlb_m)[-2:]:
    print(f"  worst {route}  {problem.route_crlb_m(route):>6.1f} m")

## 3. Exact D-optimality is a QUBO

For two-dimensional position, `det J` of the accumulated information is a quadratic
form in the binary selection variables. The pairwise coefficient between two cells is
the squared cross product of their gradients -- the Cauchy-Binet expansion. So the
*exact* information criterion goes into the QUBO with no approximation.


In [ ]:
exact_problem = replace(problem, information_objective="d_optimal")
linear, quadratic, constant = exact_problem.information_qubo_terms()

errors = []
for route in exact_problem.all_routes():
    x = np.array([int(b) for b in exact_problem.route_to_bitstring(route)], dtype=float)
    encoded = constant + linear @ x + x @ quadratic @ x
    errors.append(abs(encoded - np.linalg.det(exact_problem.route_fisher(route))))

print(f"max |QUBO det - true det| over all {len(errors)} routes: {max(errors):.2e}")
print("diagonal information terms (rank-one cells contribute none):", np.allclose(np.diag(quadratic), 0.0))

The price is connectivity: information couples **every pair** of time steps, not just
neighbours. That is also why the dynamic program stops being valid.


In [ ]:
from qmagopt import solve_brute_force, solve_dynamic_programming

try:
    solve_dynamic_programming(exact_problem)
except ValueError as error:
    print("DP refuses:", error)

print("brute force still works:", solve_brute_force(exact_problem).route)

Does encoding the exact criterion actually help? Solve both objectives exactly and
compare the quantity a navigator cares about.


In [ ]:
import statistics

separable_crlb, exact_crlb = [], []
for seed in range(100, 120):
    a = build_demo_problem(seed=seed)
    b = replace(a, information_objective="d_optimal")
    separable_crlb.append(a.route_crlb_m(solve_brute_force(a).route))
    exact_crlb.append(b.route_crlb_m(solve_brute_force(b).route))

print(f"median CRLB, separable surrogate: {statistics.median(separable_crlb):.2f} m")
print(f"median CRLB, exact D-optimal:     {statistics.median(exact_crlb):.2f} m")

## 4. Constraint-preserving QAOA

The penalty encoding searches all 2^15 = 32768 basis states, most of which are not
routes at all. The XY mixer stays inside the 3^5 = 243 one-hot states by construction.
Comparing raw feasibility would just be comparing search-space sizes, so the
**conditional** feasibility inside the one-hot sector is the number that matters.


In [ ]:
from qmagopt import QaoaConfig, solve_penalty_qaoa, solve_xy_qaoa

config = QaoaConfig(p=2, restarts=4, maxiter=120, objective="cvar", seed=21)
for solve in (solve_penalty_qaoa, solve_xy_qaoa):
    r = solve(problem, config)
    print(
        f"{r.method:<40} route {r.route}  cost {r.objective:6.3f}  "
        f"P(1-hot) {r.one_hot_probability:.4f}  "
        f"P(feasible | 1-hot) {r.conditional_feasible_probability:.4f}"
    )

Feasibility mass grows with circuit depth when the schedule is warm-started
layer to layer with the INTERP heuristic.


In [ ]:
from qmagopt import qaoa_depth_scan

scan = qaoa_depth_scan(problem, "xy", p_max=4, base_config=QaoaConfig(restarts=3, maxiter=150, seed=11))
for depth, result in zip(scan.depths, scan.results, strict=True):
    print(f"p={depth}  P(feasible | 1-hot) = {result.conditional_feasible_probability:.3f}")

## 5. Figures


In [ ]:
from IPython.display import Image, display

from qmagopt import solve_greedy, solve_simulated_annealing
from qmagopt.plotting import plot_magnetic_field, plot_routes

results = [
    solve_dynamic_programming(problem),
    solve_greedy(problem),
    solve_simulated_annealing(problem, sweeps=400, restarts=8),
    solve_xy_qaoa(problem, config),
]
display(Image(str(plot_magnetic_field(problem, "../results/nb_map.png"))))
display(Image(str(plot_routes(problem, results, "../results/nb_routes.png"))))

---

**What this is not.** Every instance here is solved exactly, and far faster, by the
classical dynamic program or by simulated annealing on the same QUBO. The contribution
is a defensible formulation and an honest benchmark, not evidence of quantum advantage.
